In [ ]:
!mkdir -p Sources/05.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

# HIXL llm-datadist 接口详解

llm-datadist 是 HIXL 面向 PD 分离场景的高层 KV Cache 传输库，封装了集群管理、内存注册、数据传输的完整流程。本节聚焦于 Python 接口的端到端调用流程，帮助你快速建立整体认知；具体的接口参数与用法请参考 [llm-datadist Python API 文档](https://gitcode.com/cann/hixl/tree/master/docs/zh/api/python)。

本节学习大纲如下：

- llm-datadist 的定位与架构
- Python 接口调用流程
- 开发注意事项
- 完整开发流程与示例
- llm-datadist 在 omni-infer 框架中的应用


## 1. llm-datadist 的定位与架构

### 1.1 定位

llm-datadist 是 HIXL 面向 PD 分离场景的高层 KV Cache 传输库，具有以下特点：

- 封装了集群管理、内存注册、数据传输的完整流程
- 底层基于 HIXL 单边通信，可切换传输后端
- 提供 C++ 和 Python 双语言接口
- 支持连续 Cache 和分块 Cache（Paged Attention）两种传输模式
- 支持逐层异步传输，实现计算与传输重叠

### 1.2 架构层次

<img src="./images/llm_datadist_architecture.svg" alt="llm-datadist 架构层次图" width="90%">

| 层次 | 组件 | 说明 |
| --- | --- | --- |
| 应用层 | 推理框架（vLLM/omni-infer） | 调用 llm-datadist API |
| 接口层 | llm-datadist（C++/Python） | 集群管理 + KV Cache 传输 |
| 通信层 | HIXL / ADXL | 单边通信底层 |
| 硬件层 | HCCS / RoCE / FabricMem | 物理传输链路 |

### 1.3 与 HIXL CS 模型的关系

llm-datadist 的底层基于 HIXL CS（Client-Server）单边通信模型：

- **Server（Prefill 端）**：创建 endpoint，注册并导出 KV Cache 内存，监听连接
- **Client（Decode 端）**：连接到 Prefill Server，导入远端内存，执行 Get/Put

| llm-datadist 操作 | HIXL CS 操作 | 说明 |
| --- | --- | --- |
| pull_cache / pull_blocks | Get | Decode(Client) 读取 Prefill(Server) 的 KV Cache |
| push_cache / push_blocks | Put | Client 写入 Server 的远端内存 |

单边通信的核心优势：Server 无需参与数据搬运，Prefill 节点的 NPU 可专注计算，实现计算与传输重叠。

### 1.4 支持的硬件

| 硬件型号 | 传输方式 | 说明 |
| --- | --- | --- |
| Atlas 800I A2 / A200I A2 Box | HCCS D2D | 仅支持机内 D2D |
| Atlas A3 | HCCS | 不支持 Host 内存作为远端 Cache |
| Ascend 950PR / 950DT（简称 A5）| UB / RoCE | 超节点内 UB，跨超节点 RoCE |


## 2. Python 接口调用流程

llm-datadist Python 接口的完整调用流程如下：

```
初始化 --> 集群链接 --> KV Cache 注册/分配 --> Pull/Push 传输 --> 资源释放
```

> 以下仅介绍各步骤的核心职责与关键接口名称，具体参数与完整签名请参考 [Python API 文档](https://gitcode.com/cann/hixl/tree/master/docs/zh/api/python)。

### 2.1 初始化与角色管理

通过 `LLMDataDist(role, cluster_id)` 创建实例，指定角色（`LLMRole.PROMPT` / `LLMRole.DECODER`）和集群 ID，再调用 `init(options)` 传入配置项完成初始化。`LLMConfig` 辅助类封装了常用配置项：

| 配置项 | 说明 |
| --- | --- |
| `device_id` | NPU 设备 ID |
| `listen_ip_info` | 监听 IP:Port。使用 HCCL 后端时仅 Server 端需要；使用 hixl 后端时所有节点均需配置 |
| `transfer_backend` | 传输后端（不设置时默认使用 HCCL 通信后端；设为 `"hixl"` 时使用 HIXL 传输后端，仅支持 `link_clusters()`） |
| `enable_cache_manager` | 启用 CacheManager（推荐 `True`） |
| `mem_pool_cfg` | 内存池配置（JSON 字符串）。**注意：该项已标记待废弃**，推荐生产环境使用 `register_cache()` / `register_blocks_cache()` 注册推理框架自管理的内存 |

**CacheManager** 是 Python 接口的核心，封装了 Cache 注册、分配、传输的高层操作，建议优先使用。

### 2.2 集群链接

支持两种链接方式：

- **双向链接**：P 和 D 双方都需要配置 rank\_table 并发起 `link()`，通过 `query_register_mem_status()` 轮询确认对端内存就绪，适合 1P1D 固定拓扑
- **单向链接**：仅 Client 端发起 `link_clusters()`，Server 端仅需 `listen_ip_info` 监听，适合 xPyD 灵活拓扑。Client/Server 角色与 Prefill/Decode 角色无关，可自由组合

<img src="./images/cluster_link_modes.svg" alt="集群链接模式对比图" width="90%">

### 2.3 KV Cache 注册与分配

通过 `CacheDesc` 描述 Cache 的形状（`shape`）、数据类型（`data_type`）和内存位置（`placement`），然后选择注册或分配方式：

| 方式 | 接口 | 适用场景 |
| --- | --- | --- |
| 注册已有内存 | `register_blocks_cache()` | 推理框架自管理 KV Cache 内存（如 vLLM PagedAttention） |
| 分配新内存 | `allocate_cache()` | 由 CacheManager 管理内存，适合简单场景或测试。**注意：该接口已标记待废弃**，推荐使用 `register_cache()` / `register_blocks_cache()` |

> **内存注册时序**：使用 `register_cache()` / `register_blocks_cache()` 注册内存时，必须在 `link()` / `link_clusters()` **之前**完成注册。使用 `allocate_cache()` 时则可在建链后分配（如本节示例所示）。

### 2.4 Pull/Push 传输

| 模式 | 接口 | 适用场景 |
| --- | --- | --- |
| 连续 Cache Pull | `pull_cache()` | 非 Paged Attention，Decode 端拉取 |
| 分块 Cache Pull | `pull_blocks()` | Paged Attention，支持 block 重排 |
| 连续 Cache Push | `push_cache()` | Prefill 端主动推送 |
| 分块 Cache Push | `push_blocks()` | Paged Attention，支持 block 重排 |

> **block 重排**：`src_blocks` 和 `dst_blocks` 可使用不同编号映射（如 `src=[0,1] → dst=[0,2]`），实现灵活的内存布局。

> **Push 前置条件**：`push_cache()` 和 `push_blocks()` 要求配置 `enable_remote_cache_accessible = True`（使用 hixl 后端时自动启用）。此外，`push_cache()` 的目标 CacheKey 需使用 `CacheKeyByIdAndIndex` 类型，`push_blocks()` 需使用 `BlocksCacheKey` 类型，而非普通 `CacheKey`。

### 2.5 逐层异步传输

Prefill 计算按层进行，每算完一层立即传输到 Decode 端，实现计算与传输重叠，隐藏传输延迟。通过 `transfer_cache_async()` 发起异步传输，配合自定义 `LayerSynchronizer` 控制每层传输时机。

<img src="./images/layer_async_transfer.svg" alt="逐层异步传输时序图" width="90%">

### 2.6 资源释放

正确的释放顺序：`unlink` / `unlink_clusters` --> `remove_cache_key`（如有关联） --> `unregister_cache` / `deallocate_cache` --> `finalize`

> **注意**：错误顺序（如先 Finalize 再 Unregister）会导致资源泄漏或进程崩溃，详见第 3 节。

> **废弃提示**：`remove_cache_key()` 和 `deallocate_cache()` 已标记待废弃，推荐使用 `unregister_cache()` / `unregister_blocks_cache()` 释放内存。本节示例仍使用旧接口以简化流程。


## 3. 开发注意事项

### 3.1 资源释放顺序

正确的释放顺序：Unlink → RemoveCacheKey（如有关联） → Unregister/Deallocate → Finalize

```python
# 正确的释放顺序（allocate_cache 模式）
datadist.unlink(comm_id)                    # 1. 先断链
cache_manager.remove_cache_key(cache_key)   # 2. 移除 CacheKey（如有关联）
cache_manager.deallocate_cache(cache)        # 3. 释放 Cache
datadist.finalize()                          # 4. 最后 Finalize
```

错误顺序（如先 Finalize 再 Unregister）会导致资源泄漏或进程崩溃。

> **废弃提示**：`remove_cache_key()` 和 `deallocate_cache()` 已标记待废弃，推荐使用 `unregister_cache()` / `unregister_blocks_cache()` 释放内存。

### 3.2 内存对齐要求

HCCS 传输要求注册内存首地址 **2MB 对齐**。建议使用大页内存分配，torch\_npu 默认使用大页内存，`data_ptr()` 通常已对齐。

### 3.3 常见问题排查

| 问题 | 可能原因 | 排查方法 |
| --- | --- | --- |
| 建链超时 | 网络不通、端口未开放 | `ping` 检查连通性，`telnet` 检查端口 |
| KV Cache 不存在 | 对端未注册或已注销 | 检查对端 register\_cache 调用顺序 |
| 传输数据错乱 | block 映射错误 | 检查 src\_blocks/dst\_blocks 对应关系 |
| OOM | Cache 过大 | 减小 batch\_size 或 seq\_len |
| 延迟过高 | 未使用逐层异步 | 启用 `transfer_cache_async` |
| 内存未对齐 | 非 2MB 对齐 | 使用大页内存分配 |

> 完整状态码定义与处理建议请参考 [Python API 文档](https://gitcode.com/cann/hixl/tree/master/docs/zh/api/python)。


## 4. 完整开发流程与示例

本节以 1P1D 场景下双向链接 + 连续 Cache Pull 为例，展示 llm-datadist 的端到端开发流程，帮助你将前面学习的 API 串联起来，形成完整的开发认知。

### 4.1 开发流程

llm-datadist 的完整开发流程如下：

```
1. 资源初始化模块：初始化 LLMDataDist（配置 device_id、listen_ip_info 等）
2. 集群链接：拆分 Prefill/Decode 脚本分别运行，link() / link_clusters()
3. KV Cache 管理模块：注册或分配 KV Cache 内存
   - 使用 register_cache / register_blocks_cache 时，须在建链之前完成注册
   - 使用 allocate_cache 时，可在建链之后分配（如本节示例所示）
4. 数据传输：Decode 前，Pull/Push KV Cache（pull_cache / pull_blocks / push_cache / push_blocks / transfer_cache_async）
5. 运行 P 和 D 脚本
6. 资源释放：Unlink → RemoveCacheKey（如有关联） → Unregister/Deallocate → Finalize
```

### 4.2 完整示例：Pull 模式（双向链接）

样例使用 Prefill 和 Decode 双脚本模式，分别在两个 NPU 上执行。为简化演示，样例在单节点双卡环境（device 0 和 device 1）上运行。

#### Prefill 端实现

Prefill 端主执行流程在 `pull_prefill.py` 中，执行流程如下：

初始化（配置 mem_pool_cfg） --> 构建 rank_table 并建链 --> 等待建链完成 --> 分配 KV Cache --> 等待 Decode 端拉取 --> 断链 --> 等待对端断链 --> 释放 Cache --> 析构

> 为了简化流程，样例中 Prefill 和 Decode 端通过 `torch.distributed.barrier()` 同步 Cache 分配与拉取时序。实际业务中，可通过推理框架的同步机制或 TCP 消息通知控制时序。

In [ ]:
%%writefile Sources/05.03/pull_prefill.py

from llm_datadist import (
    LLMDataDist, LLMRole, LLMConfig, CacheDesc,
    Placement, DataType, CacheKey, RegisterMemStatus
)
import os
import re
import torch
import torch_npu
import torch.distributed as dist
import json
import subprocess
import time
import datetime

PREFILL_CLUSTER_ID = 1
DECODE_CLUSTER_ID = 2
num_layers = 2
batch_size = 1
num_heads = 8
seq_len = 128
head_dim = 128


def get_npu_ids():
    out = subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout
    return [i for i in re.findall(r"\|\s*(\d+)\s+\S+\s+\|", out)]


def get_device_ip(device_id):
    result = subprocess.run(
        ["hccn_tool", "-i", str(device_id), "-ip", "-g"],
        stdout=subprocess.PIPE, text=True
    )
    for line in result.stdout.splitlines():
        if line.startswith("ipaddr:"):
            return line.split(":")[1]
    return ""


# 1. 初始化（Prefill 端）
torch.npu.set_device(0)
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
dist.init_process_group(backend="gloo", rank=0, world_size=2, timeout=datetime.timedelta(seconds=30))

llm_config = LLMConfig()
llm_config.device_id = 0
llm_config.enable_cache_manager = True
llm_config.mem_pool_cfg = '{"memory_size": 67108864}'
options = llm_config.generate_options()

datadist = LLMDataDist(LLMRole.PROMPT, cluster_id=PREFILL_CLUSTER_ID)
datadist.init(options)
cache_manager = datadist.cache_manager

npu_ids = get_npu_ids()

# 2. 构建 rank_table 并双向建链
rank_table = {
    "server_count": "1",
    "status": "completed",
    "version": "1.2",
    "server_list": [
        {
            "device": [
                {"device_id": npu_ids[0], "device_ip": get_device_ip(npu_ids[0]), "rank_id": "0"},
                {"device_id": npu_ids[1], "device_ip": get_device_ip(npu_ids[1]), "rank_id": "1"}
            ],
            "host_ip": "127.0.0.1",
            "server_id": "1"
        }
    ]
}
comm_id = datadist.link(
    comm_name="pd_link",
    cluster_rank_info={PREFILL_CLUSTER_ID: 0, DECODE_CLUSTER_ID: 1},
    rank_table=json.dumps(rank_table)
)

# 2.1 等待建链完成
while True:
    ret = datadist.query_register_mem_status(comm_id)
    if ret == RegisterMemStatus.OK:
        break
    if ret == RegisterMemStatus.FAILED:
        raise RuntimeError("link failed: register mem status FAILED")
    time.sleep(0.1)
print("Prefill: link established.")

# 3. 分配 KV Cache 并关联 CacheKey（供 Decode 端拉取）
cache_desc = CacheDesc(
    num_tensors=2 * num_layers,
    shape=[batch_size, num_heads, seq_len, head_dim],
    data_type=DataType.DT_FLOAT16,
    placement=Placement.DEVICE,
)

cache_key = CacheKey(prompt_cluster_id=PREFILL_CLUSTER_ID, req_id=0, model_id=0)
cache = cache_manager.allocate_cache(cache_desc, [cache_key])

print("Prefill: KV Cache allocated, waiting for Decode to pull...")
dist.barrier()  # 通知 Decode 端 Cache 已就绪
dist.barrier()  # 等待 Decode 端完成拉取

# 4. 释放资源
datadist.unlink(comm_id)
dist.barrier()  # 等待对端断链完成
cache_manager.remove_cache_key(cache_key)
cache_manager.deallocate_cache(cache)
datadist.finalize()
print("Prefill process completed.")

#### Decode 端实现

Decode 端主执行流程在 `pull_decode.py` 中，执行流程如下：

初始化 --> 构建 rank_table 并建链 --> 等待 Prefill 内存就绪 --> 分配 Cache --> Pull KV Cache --> 断链 --> 等待对端断链 --> 释放 Cache --> 析构

In [ ]:
%%writefile Sources/05.03/pull_decode.py

from llm_datadist import (
    LLMDataDist, LLMRole, LLMConfig, CacheDesc,
    Placement, DataType, CacheKey, RegisterMemStatus
)
import os
import re
import json
import subprocess
import time
import datetime
import torch
import torch_npu
import torch.distributed as dist

DECODE_CLUSTER_ID = 2
PREFILL_CLUSTER_ID = 1
num_layers = 2
batch_size = 1
num_heads = 8
seq_len = 128
head_dim = 128


def get_npu_ids():
    out = subprocess.run(["npu-smi", "info"], capture_output=True, text=True).stdout
    return [i for i in re.findall(r"\|\s*(\d+)\s+\S+\s+\|", out)]


def get_device_ip(device_id):
    result = subprocess.run(
        ["hccn_tool", "-i", device_id, "-ip", "-g"],
        stdout=subprocess.PIPE, text=True
    )
    for line in result.stdout.splitlines():
        if line.startswith("ipaddr:"):
            return line.split(":")[1]
    return ""


# 1. 初始化（Decode 端）
torch.npu.set_device(1)
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
dist.init_process_group(backend="gloo", rank=1, world_size=2, timeout=datetime.timedelta(seconds=30))

llm_config = LLMConfig()
llm_config.device_id = 1
llm_config.enable_cache_manager = True
llm_config.mem_pool_cfg = '{"memory_size": 67108864}'
options = llm_config.generate_options()

datadist = LLMDataDist(LLMRole.DECODER, cluster_id=DECODE_CLUSTER_ID)
datadist.init(options)
cache_manager = datadist.cache_manager

npu_ids = get_npu_ids()

# 2. 构建 rank_table 并双向建链
rank_table = {
    "server_count": "1",
    "status": "completed",
    "version": "1.2",
    "server_list": [
        {
            "device": [
                {"device_id": npu_ids[0], "device_ip": get_device_ip(npu_ids[0]), "rank_id": "0"},
                {"device_id": npu_ids[1], "device_ip": get_device_ip(npu_ids[1]), "rank_id": "1"}
            ],
            "host_ip": "127.0.0.1",
            "server_id": "1"
        }
    ]
}
comm_id = datadist.link(
    comm_name="pd_link",
    cluster_rank_info={PREFILL_CLUSTER_ID: 0, DECODE_CLUSTER_ID: 1},
    rank_table=json.dumps(rank_table)
)

# 3. 等待 Prefill 端注册内存就绪
while True:
    ret = datadist.query_register_mem_status(comm_id)
    if ret == RegisterMemStatus.OK:
        break
    if ret == RegisterMemStatus.FAILED:
        raise RuntimeError("link failed: register mem status FAILED")
    time.sleep(0.1)
print("Decode: Prefill memory ready.")

# 4. 分配 Cache 并 Pull
cache_desc = CacheDesc(
    num_tensors=2 * num_layers,
    shape=[batch_size, num_heads, seq_len, head_dim],
    data_type=DataType.DT_FLOAT16,
    placement=Placement.DEVICE,
)

cache = cache_manager.allocate_cache(cache_desc)

dist.barrier()  # 等待 Prefill 端 Cache 就绪

cache_key = CacheKey(prompt_cluster_id=PREFILL_CLUSTER_ID, req_id=0, model_id=0)
cache_manager.pull_cache(cache_key, cache, batch_index=0)
print("Decode: KV Cache pulled successfully.")

dist.barrier()  # 通知 Prefill 端拉取完成

# 5. 释放资源
datadist.unlink(comm_id)
dist.barrier()  # 等待对端断链完成
cache_manager.deallocate_cache(cache)
datadist.finalize()
print("Decode process completed.")

### 4.3 运行脚本

将 Prefill 和 Decode 脚本的启动命令统一放在 `run_pull.sh` 中，通过执行脚本一键拉起两个进程。

In [ ]:
%%writefile Sources/05.03/run_pull.sh

#!/bin/bash
set -e

cleanup() {
    kill "${prefill_pid}" 2>/dev/null || true
    kill "${decode_pid}" 2>/dev/null || true
    wait "${prefill_pid}" 2>/dev/null || true
    wait "${decode_pid}" 2>/dev/null || true
}
trap cleanup EXIT

export HCCL_INTRA_ROCE_ENABLE=1

python3 pull_prefill.py &
prefill_pid=$!

python3 pull_decode.py &
decode_pid=$!

wait "${prefill_pid}"
wait "${decode_pid}"

echo "Run success!"

再执行以下代码，拉起 Prefill 和 Decode 进程。

In [ ]:
!cd Sources/05.03 && bash run_pull.sh

### 4.4 更多示例代码

更多示例代码可参考 HIXL 仓库 `examples/python/` 目录：

- `pull_cache_sample.py`：双向链接 + 连续 Cache Pull
- `pull_blocks_sample.py`：分块 Cache Pull
- `push_blocks_sample.py`：单向链接 + 分块 Cache Push
- `transfer_cache_async_sample.py`：逐层异步传输
- `pull_blocks_xpyd_sample.py`：xPyD 多 Prefill 拉取
- `switch_role_sample.py`：角色动态切换

## 5. llm-datadist 在 omni-infer 框架中的应用

omni-infer 是面向昇腾 NPU 的推理加速框架，完全兼容 vLLM 接口，提供企业级 PD 分离调度（xPyD）能力。omni-infer 在底层原生集成 llm-datadist 实现 KV Cache 传输，框架开发者无需直接调用底层 API。

### 5.1 PD 连接器架构

omni-infer 的 `accelerators/pd/` 目录封装了 llm-datadist 的全部调用，形成 PD 连接器层：

| 组件 | 职责 |
| --- | --- |
| `llmdatadist_manager` | 管理 llm-datadist 实例生命周期，协调多节点链接 |
| `llmdatadist_connector` | 封装 KV Cache 传输操作（pull/push/transfer_async） |
| `omni_cache_connector` | Cache 管理抽象层，对接推理框架的 Cache 系统 |

这层抽象使得 omni-infer 可以同时支持 vLLM、SGLang 等不同推理框架，而 KV Cache 传输逻辑保持一致——开发者只需关注框架层面的调度逻辑，传输细节由 llm-datadist 统一处理。

### 5.2 xPyD 调度与传输

omni-infer 的 xPyD（交叉 Prefill-Decode）调度能力基于 llm-datadist 的**单向链接 + pull_blocks**模式实现：

- Global Proxy 负责请求路由，根据负载均衡策略选择 Prefill 和 Decode 节点
- Prefill 节点计算完成后，通过 llm-datadist 将 KV Cache 传输到选定的 Decode 节点
- Decode 节点通过 `link_clusters` 链接所有 Prefill 节点，按需 `pull_blocks` 拉取 KV Cache

### 5.3 完整调用链路

从用户请求到物理链路，omni-infer 的完整调用链路如下：

<img src="./images/tech_stack.svg" alt="omni-infer 完整技术栈调用链路图" width="90%">

可以看到，llm-datadist 处于 PD 连接器与 HIXL Engine 之间，向上提供 KV Cache 语义接口，向下调用 HIXL 单边通信原语，是连接框架调度与底层传输的关键桥梁。

## 课后练习

本节介绍了 llm-datadist 的定位架构、Python 接口调用流程与开发注意事项，请根据学习内容完成以下题目进行自测。

1. （判断题）`LLMRole.PROMPT` 表示 Prefill 角色，是 KV Cache 的生产者。

2. （判断题）双向链接要求 P 和 D 双方都配置 IP 列表并发起 link。

3. （单选题）以下哪个接口用于逐层异步传输？
    A. `pull_cache`
    B. `push_blocks`
    C. `transfer_cache_async`
    D. `CopyKvCache`

4. （多选题）以下哪些是 llm-datadist 的配置项？
    A. `llm.DeviceId`
    B. `llm.ListenIpInfo`
    C. `llm.TransferBackend`
    D. `llm.SyncKvCacheWaitTime`

5. （多选题）关于 Pull 和 Push 模式，以下说法正确的是？
    A. Pull 模式由 Decode 端发起
    B. Push 模式由 Prefill 端发起
    C. Pull 适合 Decode 端控制传输时机的场景
    D. Push 适合 Prefill 端计算完即推送的场景

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/05.03_answer.txt